# MCP 02 · 资源与提示词（Resource & Prompt）

MCP 三大件里，「工具」让模型**做事**（有副作用），这一课讲另外两件：
**资源**让模型**读数据**（只读、URI 定位），**提示词**把团队沉淀的 Prompt 统一分发。
两者的共同点是**都不调用大模型** —— 纯字符串 / 数据搬运，所以这一课跑得又快又不花钱。

| 装饰器 | 用途 | 客户端调用 | 返回 |
|---|---|---|---|
| `@mcp.tool` | 执行操作（计算 / 查询 / 写入，有副作用） | `call_tool(name, args)` | 执行结果 |
| `@mcp.resource` | 暴露**只读数据** | `read_resource(uri)` | 数据内容 |
| `@mcp.prompt` | 提供**提示词模板** | `get_prompt(name, args)` | 拼接好的消息 |

> 本 notebook 合并自 4 个源文件（归档在 `Agent/_py_source/05_mcp/`，只读）：
>
> | 源文件 | 角色 |
> |---|---|
> | `03_资源.py` | 资源的**课案原版**：2 个资源、46 行 |
> | `03_资源_jxsd.py` | 资源的**完整版**：6 个资源（含真查 PostgreSQL 的 3 个） |
> | `04_提示词.py` | 提示词的**课案原版**：2 个模板、46 行 |
> | `04_提示词_jxsd.py` | 提示词的**完整版**：4 个模板（含 `Message` 多角色写法） |

**一条链路走到底**：本 notebook 自己写出服务端脚本 → 自己起服务 → 客户端读资源 / 拉提示词
→ 最后一个格子把服务关掉。不需要你另开窗口先跑别的文件。

## 运行条件

| 项 | 说明 |
|---|---|
| 🔴 运行档位 | **需外部服务** —— 本 notebook 会在后台起一个真的 MCP 服务端（HTTP，端口 **8120**），跑完自己关掉 |
| 依赖 | `fastmcp`（venv 已装 3.4.7）、`uvicorn`、`psycopg` —— 全部已装，不需要 `pip install` |
| 密钥 | **不需要模型密钥**：资源和提示词都不调用大模型 |
| 数据库 | PostgreSQL（连接串来自仓库根 `.env` 的 `settings.pg_uri`） |
| 前置服务 | 无（服务由本 notebook 自己起） |
| 端口 | **8120**（同章 5 个 notebook 并发，各自分配了不同端口；课案原版里写的 8000 只作文本保留，本课不会去绑它） |
| 预计耗时 | 约 15 秒 |

> 数据库连不上也能跑完：`users://top` / `db://*` 这几个资源自带**降级分支**，
> 服务端会打印一行中文提示、返回内存里的演示数据，教学流程不会断。

## 本节地图

```mermaid
graph LR
    A["notebook 内核<br/>（本课自己就是客户端）"] -->|"① 写脚本 + stdio 拉起<br/>（原版，不用端口）"| B["server_basic_res.py<br/>server_basic_prompt.py"]
    A -->|"② 落盘"| C["server.py<br/>资源 + 提示词 + 起服务脚手架"]
    A -->|"③ 后台起进程<br/>端口 8120"| C
    A -->|"④ Client(url) 读资源 / 拉提示词"| C
    C -->|"真查 SQL"| D[("PostgreSQL<br/>settings.pg_uri")]
    A -->|"⑤ taskkill 收尾"| C
```

上图等价于下面这张表（每个格子的顺序就是它的编号）：

| # | 做什么 | 关键 API | 在哪个 cell |
|---|---|---|---|
| ① | 原版两个脚本落盘，用 **stdio** 传输拉起来 | `PythonStdioTransport` / `Client(transport)` | 1.1 / 1.2 / 1.3 |
| ② | 完整版服务端脚本落盘（资源 + 提示词 + 脚手架） | `Path.write_text` | 2.2 ~ 2.5 |
| ③ | 独立进程起服务 + 轮询端口就绪 | `subprocess.Popen` / `_port_in_use` | 2.6 |
| ④ | 客户端读资源、拉提示词 | `list_resources` / `read_resource` / `list_prompts` / `get_prompt` | 2.7 / 2.8 |
| ⑤ | 关掉服务（连子进程树一起收） | `taskkill /F /T /PID` | 2.9 |

**和上下节的衔接**：

- 上一课 `01_服务端与客户端.ipynb` 讲的是「服务端 / 客户端怎么握手、两种传输怎么选」，
  本课直接拿它的结论用：**原版走 stdio（客户端拉进程，不用端口）、完整版走 HTTP（自己起常驻服务）**。
- 下一课讲「工具」的进阶用法；资源与工具的区别只有一条：**资源是只读的**，
  所以资源里绝不出现写库、发邮件这类副作用，也正因如此它才敢用 URI 这样「像文件路径」的寻址方式。

## 0. 环境引导

notebook 的内核目录默认是**它自己所在的文件夹**（`Agent/05_mcp/`），而本项目所有代码都写
`from config import settings`，`config.py` 在仓库根 —— 所以每个 notebook 的第一格都统一做一件事：
**把仓库根加进 `sys.path`，并 chdir 过去**。少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

顺便定义本课的两个目录：`NB_DIR`（notebook 自己所在目录，相当于脚本里的 `__file__`）
和 `WORKDIR`（本课的临时工作目录，已被 `.gitignore` 覆盖）。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

### 预期输出

```text
仓库根： F:\ProGram\Python_Base
临时目录： F:\ProGram\Python_Base\Agent\05_mcp\tmp_nb_work
```

### 0.1 前置条件自检

这四行输出就是本课的「体检报告」，分别对应本课会踩到的四类前置条件：

1. **fastmcp 版本** —— 3.x 和 2.x 的 API 形状不一样（`Message`、`uriTemplate` 都是 3.x 才有的），
   版本不对后面会出现「代码看着没错但就是报错」；
2. **PostgreSQL 通不通** —— 决定 `db://tables` 这类资源走真查询还是走降级分支；
3. **端口 8120 有没有被占** —— 被占说明上一轮的服务还没关干净（或者别的 notebook 在跑），
   这一格只是报告，真正处理在 2.6；
4. **工作目录** —— 本课所有落盘文件都放在 `tmp_nb_work/mcp_resources/` 这个**专属子目录**里。
   同章有 5 个 notebook 会并发执行，`tmp_nb_work` 是共享容器，不套一层就会互相踩文件。

这里定义的 `_port_in_use()` 后面还会被 2.6 用到：既做「体检」，也做「起服务后等端口就绪」的轮询探针。

In [ ]:
# ---------- 0.1 前置条件自检：缺什么就用中文说清楚，别让后面每一格报看不懂的错 ----------
import socket
import time

import fastmcp
import psycopg
from config import settings

# ---- 本课全局常量：改端口只改这里（服务端脚本里还会再写一份，2.5 会断言两份一致）----
HTTP_HOST = "127.0.0.1"
HTTP_PORT = 8120                    # 同章 5 个 notebook 并发，端口各自分配：本课固定 8120
MCP_PATH = "/mcp"
MB_DIR = WORKDIR / "mcp_resources"  # 本课专属子目录：同章并发执行时不会踩别人的文件
MB_DIR.mkdir(parents=True, exist_ok=True)


def _port_in_use(host: str, port: int) -> bool:
    """探测端口是否已被监听：判断外面是不是已经有一个同类服务在跑。"""
    sock = socket.socket()
    sock.settimeout(0.5)          # 探测用短超时，避免脚本卡在 connect 上
    try:
        sock.connect((host, port))
        return True               # 能连上就说明有人监听
    except OSError:
        return False              # 连不上不算错误，只表示「没人监听」
    finally:
        sock.close()              # 无论成败都要关，防止 socket 泄漏


print("前置条件自检")
print("-" * 60)
print(f"① fastmcp        ：{fastmcp.__version__}（3.x 的 Message / uriTemplate 与 2.x 不同）")

try:
    with psycopg.connect(settings.pg_uri, connect_timeout=3) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT table_schema, table_name FROM information_schema.tables "
                        "WHERE table_schema NOT IN ('pg_catalog', 'information_schema') "
                        "ORDER BY table_schema, table_name")
            _tables = cur.fetchall()
    print(f"② PostgreSQL     ：可连接，业务表 {len(_tables)} 张 → {[t[1] for t in _tables]}")
except Exception as exc:      # noqa: BLE001 —— 连不上不算失败：服务端自带降级分支
    print(f"② PostgreSQL     ：连不上（{type(exc).__name__}）→ db:// 资源会返回内存演示数据")

_port_state = "已被占用（2.6 会直接连它，不重复起）" if _port_in_use(HTTP_HOST, HTTP_PORT) else "空闲"
print(f"③ 端口 {HTTP_PORT}     ：{_port_state}")
print(f"④ 工作目录        ：{MB_DIR.relative_to(ROOT)}（本课落盘文件都放这里）")

### 预期输出

```text
前置条件自检
------------------------------------------------------------
① fastmcp        ：3.4.7（3.x 的 Message / uriTemplate 与 2.x 不同）
② PostgreSQL     ：可连接，业务表 6 张 → ['checkpoint_blobs', 'checkpoint_migrations', 'checkpoint_writes', 'checkpoints', 'store', 'store_migrations']
③ 端口 8120     ：空闲
④ 工作目录        ：Agent\05_mcp\tmp_nb_work\mcp_resources（本课落盘文件都放这里）
```

> ② 显示「连不上」也照样能跑完：`db://` 那几个资源会走降级分支。
> ③ 显示「已被占用」说明上一轮的服务没关干净 —— 2.6 会直接连它，不重复起。

## 1. 课案原版：最短实现（46 行 + 46 行）

课案原版是**两个各自独立的 46 行脚本**，一个只讲资源、一个只讲提示词：

- `03_资源.py`：2 个资源 —— 一个固定 URI（`config://app-info`）、一个带参数的动态 URI（`db://users/{user_id}`）；
- `04_提示词.py`：2 个模板 —— 一个返回字符串、一个返回**消息列表**（`list[dict]`）。

两个脚本的结尾都是一样的骨架：

```python
def run_stdio():
    mcp.run()                       # 默认 transport=stdio：用标准输入输出通信

if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "http":
        mcp.run(transport="http", host="127.0.0.1", port=8000, path="/mcp")
    else:
        run_stdio()
```

也就是说：**不传参数时它是 stdio 服务，`python 03_资源.py http` 时才是 HTTP 服务**。

本节把这两个脚本**原样**写进临时目录（一个字都不改，包括那句 `port=8000` ——
它是 `http` 分支里的死代码，本课走的是 stdio 分支，不会去绑这个端口），
然后演示 stdio 这种传输的特点：

| 传输 | 服务进程谁来起 | 占端口吗 | 适合 |
|---|---|---|---|
| **stdio** | **客户端**（把脚本当子进程拉起来） | 不占 | 本机、单客户端、随用随停 |
| **http** | 你自己（常驻服务） | 占（本课用 8120） | 多人 / 多客户端共享、要单独部署 |

### 1.1 原版服务端落盘（两个 46 行脚本，逐字保留）

下面两个变量装的就是课案原版的全文。用 `write_text` 落盘后，它们就是两个**能被别的程序启动的真脚本**。

In [ ]:
# ---------- 1.1 原版服务端：两个课案脚本原样落盘 ----------
BASIC_RES_PY = MB_DIR / "server_basic_res.py"
BASIC_PROMPT_PY = MB_DIR / "server_basic_prompt.py"

BASIC_RES_CODE = r'''
import sys

from fastmcp import FastMCP

mcp = FastMCP(name="资源演示")


# ---------- 静态资源：固定 URI ----------
@mcp.resource("config://app-info")
def app_info() -> str:
    """应用的基础信息（只读配置）"""
    return "应用名称：Agent 课案演示；版本：1.0.0"


# ---------- 动态资源：URI 模板 {param} ----------
@mcp.resource("db://users/{user_id}")
def user_profile(user_id: str) -> str:
    """按用户 ID 读取用户资料"""
    return f"用户 {user_id}：小红，上海，VIP3 会员"


def run_stdio():
    mcp.run()


if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "http":
        mcp.run(transport="http", host="127.0.0.1", port=8000, path="/mcp")
    else:
        run_stdio()
'''

BASIC_PROMPT_CODE = r'''
import sys

from fastmcp import FastMCP

mcp = FastMCP(name="提示词演示")


# ---------- 简单提示词 ----------
@mcp.prompt
def code_review(code: str) -> str:
    """代码审查提示词模板：参数会自动填充进模板"""
    return f"请审查以下代码，从 可读性 / 性能 / 安全 三个角度给出意见：\n\n{code}"


# ---------- 多消息提示词（返回消息列表，可包含角色） ----------
@mcp.prompt
def translate(text: str, target_lang: str = "英文") -> list:
    """翻译提示词：system 设定角色，user 给任务"""
    return [
        {"role": "system", "content": "你是专业翻译，译文自然流畅。"},
        {"role": "user", "content": f"把下面的内容翻译成{target_lang}：\n{text}"},
    ]


def run_stdio():
    mcp.run()


if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "http":
        mcp.run(transport="http", host="127.0.0.1", port=8000, path="/mcp")
    else:
        run_stdio()
'''

for _script in (BASIC_RES_PY, BASIC_PROMPT_PY):
    _script.with_suffix(".log").unlink(missing_ok=True)   # 清掉上一轮的服务端日志，免得看混淆

BASIC_RES_PY.write_text(BASIC_RES_CODE, encoding="utf-8")
BASIC_PROMPT_PY.write_text(BASIC_PROMPT_CODE, encoding="utf-8")
print(f"已写入：{BASIC_RES_PY.relative_to(ROOT)}（{len(BASIC_RES_CODE.splitlines())} 行）")
print(f"已写入：{BASIC_PROMPT_PY.relative_to(ROOT)}（{len(BASIC_PROMPT_CODE.splitlines())} 行）")

### 预期输出

```text
已写入：Agent\05_mcp\tmp_nb_work\mcp_resources\server_basic_res.py（31 行）
已写入：Agent\05_mcp\tmp_nb_work\mcp_resources\server_basic_prompt.py（34 行）
```

### 1.2 用 stdio 传输把原版资源服务端拉起来

三个必须说清楚的点：

1. **`python_cmd=sys.executable`** —— 默认值就是当前解释器，但这里显式写出来提醒一件事：
   本机 PATH 里的 `python` 是 Windows Store 的占位符（执行了**没有任何输出也不报错**），
   服务端脚本必须由 venv 的解释器来跑。
2. **`cwd=str(ROOT)`** —— 服务端子进程的工作目录也必须是仓库根，否则它 `from config import settings` 会失败。
3. **`log_file=...`** —— 服务端的 stderr 被重定向到文件。stdio 的 stdout 是协议通道，
   日志绝不能往那上面写；落到文件里，服务端一崩就能去文件里看真因（本机实测：少一行 `import sys`
   的脚本会在这里留下 `NameError` 的完整 traceback）。

> `list_resources()` 只有**固定资源**（URI 里没有 `{}`）；
> 带参数的动态资源在 `list_resource_templates()` 里，字段名是 **`uriTemplate`**（不是 `uri`）。

In [ ]:
# ---------- 1.2 客户端：用 stdio 传输把原版资源服务端拉起来（不占端口）----------
import asyncio

from fastmcp import Client
from fastmcp.client.transports import PythonStdioTransport

# 子进程环境：中文 Windows 必须 UTF-8；本机 Clash 会拦回环请求，必须设 NO_PROXY
SUB_ENV = {**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8",
           "NO_PROXY": "127.0.0.1,localhost"}


async def demo_basic_resources() -> None:
    transport = PythonStdioTransport(
        str(BASIC_RES_PY),
        python_cmd=sys.executable,                    # 必须用 venv 解释器（PATH 里的是占位符）
        cwd=str(ROOT),                                # 让子进程也站在仓库根
        env=SUB_ENV,
        keep_alive=False,                             # 退出 async with 就收掉子进程，不留孤儿进程
        log_file=BASIC_RES_PY.with_suffix(".log"),    # 服务端 stderr 落文件，不刷屏
    )
    async with Client(transport) as client:
        fixed = await client.list_resources()
        templates = await client.list_resource_templates()
        print(f"固定资源（{len(fixed)} 个）：")
        for r in fixed:
            print(f"  {r.uri} → {r.name} - {r.description}")
        print(f"动态资源（{len(templates)} 个，在 list_resource_templates 里）：")
        for t in templates:
            print(f"  {t.uriTemplate} → {t.name} - {t.description}")
        info = await client.read_resource("config://app-info")
        print(f"\n读 config://app-info → {info[0].text}")
        profile = await client.read_resource("db://users/u-7")
        print(f"读 db://users/u-7  → {profile[0].text}")


# 【notebook 改写】脚本里这里是 asyncio.run(demo_basic_resources())。
# notebook 内核里**已经有一个正在跑的事件循环**，再调 asyncio.run() 会直接报
#     RuntimeError: asyncio.run() cannot be called from a running event loop
# 所以本 notebook 里的异步调用统一改成**顶层 await**（IPython 的 autoawait，本机已实测）。
await demo_basic_resources()
print("stdio 子进程已随 async with 退出而关闭（keep_alive=False）")

### 预期输出

```text
固定资源（1 个）：
  config://app-info → app_info - 应用的基础信息（只读配置）
动态资源（1 个，在 list_resource_templates 里）：
  db://users/{user_id} → user_profile - 按用户 ID 读取用户资料

读 config://app-info → 应用名称：Agent 课案演示；版本：1.0.0
读 db://users/u-7  → 用户 u-7：小红，上海，VIP3 会员
stdio 子进程已随 async with 退出而关闭（keep_alive=False）
```

### 1.3 同样把原版提示词服务端拉起来 —— 顺便撞一次版本坑

这一段是**本课最值钱的一次现场翻车**。原版 `04_提示词.py` 的多角色模板返回的是
`[{"role": ..., "content": ...}]` 这种 **`list[dict]`**，在 FastMCP 3.x 上：

- **注册不报错**（`list_prompts()` 里看得到 `translate`）；
- **调用时才炸**，客户端收到一条 `McpError`：

```text
Error rendering prompt 'translate': messages[0] must be Message or str, got dict.
Use Message({'role': 'system', 'content': '...'}) to wrap the value.
```

所以本格把这次调用包在 `try/except` 里 —— **不是**为了藏错误，而是要把错误原文打出来给你看：
这正是「课案原版 → 完整版」要修的第一处。修法在第 2 节：用 `fastmcp.prompts.Message` 包装每一段消息。

> 还有一个协议层面的坑：MCP 的 `PromptMessage.role` **只允许 `user` / `assistant`，没有 `system`**。
> 原版写的 `{"role": "system"}` 就算包成 `Message` 也会被拒 —— 角色铺垫要改用 `assistant` 说一句话来完成（见 2.4）。

In [ ]:
# ---------- 1.3 客户端：拉原版提示词服务端（translate 会报错，正好把错误原文打出来）----------
async def demo_basic_prompts() -> None:
    transport = PythonStdioTransport(
        str(BASIC_PROMPT_PY),
        python_cmd=sys.executable,
        cwd=str(ROOT),
        env=SUB_ENV,
        keep_alive=False,
        log_file=BASIC_PROMPT_PY.with_suffix(".log"),
    )
    async with Client(transport) as client:
        prompts = await client.list_prompts()
        print(f"可用提示词（{len(prompts)} 个，description 就是函数 docstring）：")
        for p in prompts:
            print(f"  {p.name}: {p.description}")

        one = await client.get_prompt("code_review", {"code": "def add(a,b):\n    return a+b"})
        print(f"\n生成的提示词：\n{one.messages[0].content.text}")

        try:
            two = await client.get_prompt(
                "translate", {"text": "工具调用是 Agent 的基础能力。", "target_lang": "英文"}
            )
            print(f"\n翻译模板共 {len(two.messages)} 条消息")
        except Exception as exc:      # noqa: BLE001 —— 原版这段在 3.x 就是会报错，正好当教学现场
            print(f"\n⚠️  原版 translate 调用失败：{type(exc).__name__}")
            print(f"   {exc}")
            print("   → 修法见 2.4：改用 Message 对象，并把 system 换成 assistant。")


# 同样是顶层 await（原因见 1.2 的改写说明）
await demo_basic_prompts()

### 预期输出

```text
可用提示词（2 个，description 就是函数 docstring）：
  code_review: 代码审查提示词模板：参数会自动填充进模板
  translate: 翻译提示词：system 设定角色，user 给任务

生成的提示词：
请审查以下代码，从 可读性 / 性能 / 安全 三个角度给出意见：

def add(a,b):
    return a+b

⚠️  原版 translate 调用失败：McpError
   Error rendering prompt 'translate': messages[0] must be Message or str, got dict. Use Message({'role': 'system', 'content': '你是专业翻译，译文自然流畅。'}) to wrap the value.
   → 修法见 2.4：改用 Message 对象，并把 system 换成 assistant。
```

## 2. 完整版：一个服务端，6 种资源 + 4 个提示词模板

完整版把「资源」和「提示词」合到**一个服务端进程**里 —— 真实项目就是这么干的：
一个服务端对外提供它这一块的全部能力，客户端连一次就全拿到。

| 小节 | 源文件 | 补了什么 |
|---|---|---|
| 2.2 | `03_资源_jxsd.py` | 连接串从 `settings` 拼（绝不硬编码口令）、连接失败降级、`connect_timeout` |
| 2.3 | `03_资源_jxsd.py` | 6 个资源：固定 / 单参数 / **多级动态** / 真查 PostgreSQL ×3，含表名注入防护 |
| 2.4 | `04_提示词_jxsd.py` | 4 个模板、参数默认值、`Message` 多角色写法（修掉 1.3 的版本坑） |
| 2.5 | 两者 | 落盘成 `server.py` |
| 2.6 ~ 2.9 | 两者 | 后台起进程、客户端两种调用、最后关服务 |

### 2.1 资源的 URI 设计规范（先看规则，再看实现）

资源的核心不是函数名，而是 **URI**（scheme 和 path 都能自定义，建议用有意义的命名体现资源类型）：

| 形态 | 说明 | 课案示例 | 完整版里的实现 |
|---|---|---|---|
| `scheme://path` | 固定资源（无参数） | `config://app`、`menu://main` | `config://app` |
| `scheme://path/{参数}` | 动态资源（单参数） | `file://docs/{filename}` | `file://docs/{filename}` |
| `scheme://{a}/{b}` | 多级动态（多参数） | `api://users/{userId}/posts/{postId}` | 同名落地 |
| 查询类 | 把后端数据当资源读 | `users://top/{limit}` | 真连 PostgreSQL 跑 SQL |

实现上有两条铁律：

1. **URI 模板里的 `{参数}` 必须和函数参数名完全一致**，否则 FastMCP 在**注册时**就报错；
2. **资源是只读语义**：完整版的 `query_postgres()` 刻意不提供 commit、也不接受多语句，
   查完直接关连接 —— 资源接口绝不能被拿来当写入口。

另外：函数**返回什么类型，客户端就读到什么类型**。返回 `str` → `TextResourceContents`（`text/plain`）；
返回 `dict` / Pydantic 模型 → FastMCP 自动转 JSON 并标 `application/json`。

### 2.2 服务端（上）：配置与数据库访问层

这一段对应源文件 `03_资源_jxsd.py` 的第一部分，四件事：

1. `from config import settings` —— 全仓库唯一的配置入口，**不另建 `conf.py`、不散落 `os.environ`**；
2. `PG_URI` 从 `settings` 拼出来 —— 课案里的 `postgresql://<用户名>:<口令>@...` 是字面量，
   口令写进代码就等于把凭据提交进仓库，必须替换掉；
3. `FALLBACK_USERS` —— 连不上库时用的内存演示数据，保证教学流程不断；
4. `query_postgres()` —— 连接带 `connect_timeout=3`（**必给**，否则库没起时客户端会一直干等），
   出错只回「异常类型 + 一句话」，**绝不回完整连接串**（里面有口令）。

服务端脚本分三段拼出来，所以这一格先只定义第一段的文本：

In [ ]:
# ---------- 2.2 服务端脚本（上）：导入 + 端口常量 + 数据库访问层 ----------
SERVER_RESOURCES = r'''
import asyncio
import json
import logging
import re
import socket
import sys
import threading
import time
from pathlib import Path

import uvicorn
from fastmcp import FastMCP

# Message 是 FastMCP 3.x 里「一条消息」的载体：返回多角色提示词时必须用它，
# 不能再返回 [{"role": ..., "content": ...}] 这种 dict（3.x 会在渲染时直接报错）。
from fastmcp.prompts import Message

# 让这份脚本「放在哪都能单独跑」：向上找到有 config.py 的仓库根，塞进 sys.path。
# （notebook 起进程时给了 cwd=仓库根，但单独 python server.py 时也应该自己找得到。）
REPO_ROOT = Path(__file__).resolve().parent
while not (REPO_ROOT / "config.py").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("没找到 config.py：本文件必须放在 Python_Base 仓库内")
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# fastmcp 的 INFO 日志会往 stderr 刷，压到 ERROR —— 日志通道要留给真正的异常
logging.getLogger("fastmcp").setLevel(logging.ERROR)

from config import settings

# 本服务监听 8120：同章 5 个 notebook 并发，端口各自分配，不要用 8000 / 9000
HTTP_HOST = "127.0.0.1"
HTTP_PORT = 8120
MCP_PATH = "/mcp"

mcp = FastMCP("资源演示 🚀")


# ================================================================
# 一、数据库连接：连接串从 settings 拼，绝不硬编码
# ================================================================
# 规范铁律：不硬编码任何密钥、口令、连接串。课案里写的
#     postgresql://<用户名>:<口令>@127.0.0.1:5432/langgraph
# 这种字面量必须全部替换掉——口令写进代码就等于把凭据提交进仓库。
#
# 两种拼法都行：
#   拼法 A（推荐）：直接用 config.py 封装好的 property
#       settings.pg_uri        → LangGraph 用的库（本机实测有 6 张表）
#       settings.postgres_url  → 分字段拼出来的 URL
#       settings.postgres_dsn  → psycopg3 的 key=value 形式
#   拼法 B（本文用的写法，把拼装过程显式写出来，方便看清每个字段来自哪里）：
#       f"postgresql://{settings.postgres_user}:{settings.postgres_password}..."
PG_URI = settings.pg_uri or (
    f"postgresql://{settings.postgres_user}:{settings.postgres_password}"
    f"@{settings.postgres_host}:{settings.postgres_port}/{settings.postgres_db}"
)

# 连接失败的降级数据（只在连不上库时使用，避免资源调用直接抛异常）
FALLBACK_USERS = [
    {"name": "张三", "level": 99},
    {"name": "李四", "level": 85},
    {"name": "王五", "level": 72},
]


def query_postgres(sql: str, params: tuple = ()) -> tuple[bool, list | str]:
    """执行一条只读 SQL，返回 (是否成功, 行列表 或 中文错误信息)。

    资源是**只读**语义，所以这里刻意不提供 commit、也不接受多语句；
    psycopg 默认就是「显式事务」，查询完直接关连接不会有副作用。
    连接串里的口令绝不打印，出错信息也只回中文提示。
    """
    try:
        import psycopg
    except ImportError:
        return False, "未安装 psycopg，无法查询 PostgreSQL（安装：uv add psycopg）"

    try:
        # connect_timeout 必给：否则库没起时资源调用会一直吊着，客户端只能干等
        with psycopg.connect(PG_URI, connect_timeout=3) as conn:
            with conn.cursor() as cur:
                cur.execute(sql, params)
                return True, cur.fetchall()
    except Exception as exc:
        # 只回异常类型 + 一句话，不回完整连接串（里面含口令）
        return False, f"数据库不可用（{type(exc).__name__}）"
'''

print(f"SERVER_RESOURCES 已就绪：{len(SERVER_RESOURCES.splitlines())} 行（还没落盘，2.5 统一写）")

### 预期输出

```text
SERVER_RESOURCES 已就绪：89 行（还没落盘，2.5 统一写）
```

### 2.3 服务端（中）：六个资源

| 资源 | 形态 | 实现要点 |
|---|---|---|
| `config://app` | 固定 | 原样返回一段 JSON 文本（返回 `str` 就是 `text/plain`，不再自动标 JSON） |
| `file://docs/{filename}` | 单参数 | 查不到**不抛异常**，回一句「文件 xxx 不存在」——资源读不到是常态 |
| `api://users/{user_id}/posts/{post_id}` | 多级动态 | 一个 `{}` 对应一个函数参数，顺序随意、名字必须对上 |
| `users://top/{limit}` | 查询类 | 真连接 + 真 SQL（用 `VALUES` 造内联表）+ 真 `ORDER BY`；连不上就打印提示并回内存数据 |
| `db://tables` | 查询类 | 真查 `information_schema.tables`（本来就是个普通查询，没有业务参数 |
| `db://table/{table}/rows/{limit}` | 查询类 | **注入防护**：表名走「正则白名单 + 存在性校验」双重把关 |

最后那个资源是本节的重点，因为**表名不能像参数那样用 `%s` 占位**（表名是标识符、不是值），
只能拼进 SQL —— 于是它就成了资源类接口最典型的注入点：

```text
db://table/users; DROP TABLE users--/rows/10
```

完整版的做法是：先用 `_IDENT_RE` 正则拒绝掉这种表名（**连数据库都不碰**），
再查一次 `information_schema` 确认表真的存在，最后才拼串 + 用整数拼 `LIMIT`。2.7 会真发一次注入尝试给你看结果。

In [ ]:
# ---------- 2.3 服务端脚本（中）：六个资源（固定 / 单参数 / 多级动态 / 数据库查询类）----------
SERVER_RESOURCES += r'''

# ================================================================
# 二、① 固定资源：URI 里没有任何参数
# ================================================================
# 函数返回什么类型，客户端读到的就是什么类型：
#   - 返回 str  → MCP 包成 TextResourceContents（mimeType=text/plain）
#   - 返回 dict/BaseModel → FastMCP 自动转 JSON 并标 mimeType=application/json
# 课案为了演示「原样返回字符串」，直接返回了一段 JSON 文本；这里保持课案写法。
@mcp.resource("config://app")
def get_app_config() -> str:
    """返回应用配置信息"""
    return """
    {
      "app_name": "计算器服务",
      "version": "1.0.0",
      "max_precision": 10
    }
    """


# ================================================================
# 三、② 动态资源：URI 模板里带 {参数}，客户端填具体值再读
# ================================================================
# 关键点：@mcp.resource("file://docs/{filename}") 里的 {filename}
# 必须和函数参数名**完全一致**，否则 FastMCP 在注册时就会报错。
# 客户端侧看不到这个资源出现在 list_resources() 里，
# 而是出现在 list_resource_templates() 里（URI 模板列表）。
@mcp.resource("file://docs/{filename}")
def get_file_content(filename: str) -> str:
    """根据文件名返回文档内容"""
    docs = {
        "readme": "# 计算器服务\n\n提供基础数学运算的 MCP 服务。",
        "changelog": "## v1.0.0\n- 支持加减乘除四则运算",
    }
    # 查不到时不抛异常，回一句中文提示——资源读不到内容是很常见的正常情况，
    # 让模型看到「文件不存在」比让连接报错更有用。
    return docs.get(filename, f"文件 '{filename}' 不存在")


# 课案 URI 设计规范里的第三种：scheme://{a}/{b} 多级动态（多参数）。
# 每个 {xxx} 都对应一个函数参数，顺序无所谓，名字必须对上。
@mcp.resource("api://users/{user_id}/posts/{post_id}")
def get_user_post(user_id: str, post_id: str) -> str:
    """多级动态资源示例：读取某个用户的某篇帖子"""
    return json.dumps(
        {"user_id": user_id, "post_id": post_id, "title": f"{user_id} 的第 {post_id} 篇帖子"},
        ensure_ascii=False,
    )


# ================================================================
# 四、③ 数据库查询类资源（本节重点）
# ================================================================
# 课案原文这个资源是「查内存 list」：
#     users = [{"name": "张三", "level": 99}, ...]
#     return json.dumps(users[:int(limit)])
# 它想讲的其实是「资源可以动态查询后端数据」。这里把它换成**真查 PostgreSQL**，
# 顺带把三个生产上必须处理的点补上：连接失败降级、排序下推到 SQL、表名注入防护。
@mcp.resource("users://top/{limit}")
def get_top_users(limit: int) -> str:
    """查询排名前 N 的用户数据"""
    limit = max(1, min(int(limit), 50))     # 兜底：客户端可能传 0 / 负数 / 超大值

    # 本机 langgraph 库里没有业务用户表，所以用 SQL 的 VALUES 构造一张内联表再 ORDER BY。
    # 这依然是「真连接 + 真 SQL + 真排序」——把它换成你的业务表就是生产代码：
    #     SELECT name, level FROM users ORDER BY level DESC LIMIT %s
    ok, rows = query_postgres(
        "SELECT name, level FROM (VALUES ('张三', 99), ('李四', 85), ('王五', 72)) "
        "AS t(name, level) ORDER BY level DESC LIMIT %s",
        (limit,),
    )

    if not ok:
        # 降级路径：打印中文提示，返回内存演示数据，保证教学流程不断
        print(f"\n⚠️  {rows}")
        print("   → users://top 已降级为内存演示数据。")
        print(f"   请检查 .env 里的 PG_URI / POSTGRES_* 配置，以及 PostgreSQL 是否已启动。")
        return json.dumps(FALLBACK_USERS[:limit], ensure_ascii=False)

    return json.dumps([{"name": r[0], "level": r[1]} for r in rows], ensure_ascii=False)


@mcp.resource("db://tables")
def list_db_tables() -> str:
    """列出当前数据库里所有业务表（真查 information_schema）"""
    # information_schema 是 PostgreSQL 的系统视图，列出所有 schema / table；
    # 排除 pg_catalog 与 information_schema 自身，剩下的才是业务表。
    # 查询本身不拼业务参数，只读系统视图，所以这段没有任何注入风险。
    ok, rows = query_postgres(
        "SELECT table_schema, table_name FROM information_schema.tables "
        "WHERE table_schema NOT IN ('pg_catalog', 'information_schema') "
        "ORDER BY table_schema, table_name"
    )
    if not ok:
        print(f"\n⚠️  {rows} → db://tables 无法读取，返回中文提示。")
        return "数据库不可用：请确认 PostgreSQL 已启动，且 .env 中的 PG_URI 正确。"

    tables = [{"schema": r[0], "table": r[1]} for r in rows]
    return json.dumps({"count": len(tables), "tables": tables}, ensure_ascii=False, indent=2)


# 表名不能像参数那样用 %s 占位（表名是标识符不是值，占位符会被当成字符串字面量）。
# 所以只能拼进 SQL —— 拼之前必须**白名单校验**，这是资源类接口最典型的注入点：
#   db://table/users; DROP TABLE users--/rows/10 这种 URI 就是冲这里来的。
_IDENT_RE = re.compile(r"^[A-Za-z_][A-Za-z0-9_]{0,62}$")


@mcp.resource("db://table/{table}/rows/{limit}")
def get_table_rows(table: str, limit: int) -> str:
    """查看某张表的前 N 行（表名走严格白名单校验，防 SQL 注入）"""
    if not _IDENT_RE.match(table):
        # 直接拒绝，绝不把可疑字符串送进数据库
        return f"表名非法：{table!r}。只允许字母/数字/下划线，且以字母或下划线开头。"

    limit = max(1, min(int(limit), 20))

    # 只允许读「当前库里真实存在的表」，进一步收窄攻击面：
    # 先查 information_schema 确认存在，再把名字拼进 SQL。
    # 表名不能走 %s 占位（标识符不是值），必须拼串 —— 所以拼之前做「正则 + 存在性」双重校验。
    ok, rows = query_postgres(
        "SELECT 1 FROM information_schema.tables "
        "WHERE table_schema = 'public' AND table_name = %s",
        (table,),
    )
    if not ok:
        print(f"\n⚠️  {rows} → db://table 无法读取，返回中文提示。")
        return "数据库不可用：请确认 PostgreSQL 已启动，且 .env 中的 PG_URI 正确。"
    if not rows:
        return f"表 public.{table} 不存在。可先读 db://tables 看有哪些表。"

    # 表名已通过正则 + 存在性双重校验，这里拼接是安全的；LIMIT 用整数拼接同样安全
    ok, data = query_postgres(f'SELECT * FROM public."{table}" LIMIT {limit}')
    if not ok:
        return data

    # rowcount 拿不到，用 cursor.description 才能拿列名——这里简化成按序号列出，
    # 保持示例聚焦在「资源 + 数据库」本身。
    return json.dumps(
        {"table": table, "count": len(data), "rows": [list(r) for r in data]},
        ensure_ascii=False,
        default=str,      # datetime / UUID 等类型转字符串，否则 json 会报错
        indent=2,
    )
'''

print(f"SERVER_RESOURCES 现在共 {len(SERVER_RESOURCES.splitlines())} 行，"
      f"其中 @mcp.resource {sum(1 for ln in SERVER_RESOURCES.splitlines() if ln.startswith('@mcp.resource'))} 个")

### 预期输出

```text
SERVER_RESOURCES 现在共 233 行，其中 @mcp.resource 6 个
```

### 2.4 服务端（下）：四个提示词模板

为什么要费力把 Prompt 放到服务端？因为**写 Prompt 最好的人通常只有一两个**：
把它沉淀成 MCP 服务，所有人（以及所有 Agent）拉下来就能用；改 Prompt 只改服务端一处，
不用通知每个人去更新本地代码。这是 Prompt 服务化的真正价值。

完整版实现四个模板，把原版的坑一个个补掉：

| 模板 | 参数 | 要点 |
|---|---|---|
| `code_review` | `language`、`code`（必填） | 审查维度写死在服务端，客户端拿到的一定是团队规定的那套 |
| `writing_assistant` | `topic` 必填、`style="正式"` | **带默认值的参数不用传**，服务端会用默认值填 |
| `math_tutor` | `question` 必填、`difficulty="中等"` | 业务规则（难度分支）放服务端，客户端永远拿到合规提示词 |
| `translate` | `text`、`target_lang="英文"` | 返回**两条** `Message`：`assistant` 铺垫角色 + `user` 给任务 |

⚠️ 两个版本坑（1.3 已经撞过一次，这里是修法）：

1. **`list[dict]` 会被 FastMCP 3.x 拒绝**，必须用 `fastmcp.prompts.Message` 包装：
   `Message(role="assistant", content="...")`；
2. **MCP 的 `PromptMessage.role` 只允许 `user` / `assistant`**（没有 `system`），
   所以「角色设定」要么折进正文（见 `writing_assistant`），要么用 `assistant` 说一句话来铺垫
   （见 `translate`）—— 这两条路都比硬塞一个 `system` 更稳。

另外：**Prompt 本身不调用大模型**，它只负责「拼字符串 / 拼消息」，发给哪个 LLM 由客户端决定。

In [ ]:
# ---------- 2.4 服务端脚本（下）：四个提示词模板 ----------
SERVER_PROMPTS = r'''

# ================================================================
# 五、提示词模板：把团队沉淀的 Prompt 放到服务端
# ================================================================
# ---------- 1. 基础提示词模板（返回字符串） ----------
# @mcp.prompt 后面不写名字时，函数名就是提示词名（code_review）。
# 想改名写 @mcp.prompt(name="代码审查")。
@mcp.prompt
def code_review(language: str, code: str) -> str:
    """代码审查提示词"""
    # 下面这段正文就是课案原文：四个审查维度写死在服务端，
    # 客户端拿到的一定是团队规定的那套，不会各写各的。
    return f"""请审查以下 {language} 代码，从以下几个方面给出建议：
1. 代码规范性
2. 性能优化
3. 安全漏洞
4. 可读性

代码：
{code}"""


# ---------- 2. 包含角色设定的提示词（返回字符串，内含角色设定） ----------
# 注意 style: str = "正式" 这个默认值：FastMCP 会把它标成「非必填参数」，
# 客户端不传 style 时就用 "正式"。—— 提示词模板的参数默认值就是这么用的。
@mcp.prompt
def writing_assistant(topic: str, style: str = "正式") -> str:
    """写作助手提示词，内含角色和写作要求"""
    return f"""你是一位{style}风格的专业写手。
请写一篇关于「{topic}」的文章，要求不少于500字。"""


# ---------- 3. 带上下文的复杂提示词（返回字符串） ----------
# 这一段体现的是「把业务规则写进模板」：难度不同，要求不同。
# 把这种分支写在服务端，客户端就永远拿到符合规范的 Prompt。
@mcp.prompt
def math_tutor(question: str, difficulty: str = "中等") -> str:
    """数学辅导提示词"""
    return f"""你是一位耐心的数学老师，学生提出了以下问题：

{question}

难度等级：{difficulty}

要求：
- 先给出解题思路，再给出详细步骤
- 用通俗易懂的语言讲解
- 如果难度为"困难"，需要补充相关的知识点背景"""


# ---------- 4. 额外：返回「消息列表」的多角色模板（修掉原版那个 list[dict] 的写法） ----------
# 返回 list 时，列表里每一项都会变成一条消息。相比拼一个大字符串，
# 这样能把「角色铺垫」和「用户任务」分开，模型理解更稳。
#
# ⚠️ 版本坑：FastMCP 3.x 不再接受 [{"role": "system", "content": "..."}] 这种 dict，
#   会直接报 `messages[0] must be Message or str, got dict`，必须用 Message 包装。
# ⚠️ 协议坑：PromptMessage.role 只允许 user / assistant，没有 system，
#   所以这里用 assistant 先说一句来铺垫角色。
@mcp.prompt
def translate(text: str, target_lang: str = "英文") -> list:
    """翻译提示词：用 assistant 消息做角色铺垫，再用 user 消息给任务"""
    return [
        Message(role="assistant", content="我是专业翻译，译文自然流畅，只输出译文本身。"),
        Message(role="user", content=f"把下面的内容翻译成{target_lang}：\n{text}"),
    ]
'''

print(f"SERVER_PROMPTS 已就绪：{len(SERVER_PROMPTS.splitlines())} 行，"
      f"其中 @mcp.prompt {sum(1 for ln in SERVER_PROMPTS.splitlines() if ln.startswith('@mcp.prompt'))} 个")

### 预期输出

```text
SERVER_PROMPTS 已就绪：66 行，其中 @mcp.prompt 4 个
```

### 2.5 服务端脚手架与落盘：拼成一个可直接运行的 `server.py`

前两格只定义了「能力」（资源 + 提示词），真正把服务跑起来还需要第三段：**起服务脚手架**。
这一段**不能照抄原版的 `mcp.run()`** —— 它是常驻调用，放进 notebook 会把内核**永久阻塞**，
后面每一格都没机会执行。所以完整版的脚手架改成了「后台线程起 uvicorn + 主线程常驻」：

```python
def start_server_in_thread():        # 端口被占就返回 None（不是自己起的，退出时不许关）
    if _port_in_use(HTTP_HOST, HTTP_PORT): ... return None
    app = mcp.http_app(transport="streamable-http", path=MCP_PATH)
    server = uvicorn.Server(uvicorn.Config(app, host=HTTP_HOST, port=HTTP_PORT, log_level="warning"))
    thread = threading.Thread(target=server.run, daemon=True)
    ...
```

这份脚本有两种用法，本课用的是第二种：

| 用法 | 谁执行 | 生命周期 |
|---|---|---|
| `python server.py`（单独跑） | 你，在终端里 | 常驻，Ctrl+C 优雅退出 |
| **本课：`subprocess.Popen` 起独立进程** | notebook | 由 2.6 起、2.9 用 `taskkill /F /T` 收整棵进程树 |

落盘后还会做一个**一致性断言**：脚本里的端口必须和 notebook 里连的端口一致。
拼串最容易出的错就是「改了这边的 8120、忘了那边的」，让它在启动前就炸，比等客户端连不上好查得多。

这一段脚手架本身也是**能在终端里单独跑**的（`python server.py`）：那时它会常驻，
Ctrl+C 会走 `finally` 里的优雅退出（`server.should_exit = True`）。

In [ ]:
# ---------- 2.5 服务端脚本（脚手架）：后台线程起 uvicorn + 入口 ----------
SERVER_MAIN = r'''
# ================================================================
# 六、起服务脚手架 + 入口
# ================================================================
def _port_in_use(host: str, port: int) -> bool:
    """探测端口是否已被监听：判断外面是不是已经有一个同类服务在跑。"""
    sock = socket.socket()
    sock.settimeout(0.5)
    try:
        sock.connect((host, port))
        return True
    except OSError:
        return False
    finally:
        sock.close()


def start_server_in_thread():
    """后台线程起服务端；端口被占则直接连已有的那个。

    返回 None 的语义是「不是我起的」——调用方据此决定退出时要不要关掉它。
    外面已经在跑的服务绝不能动：那可能是别人正在用的服务端。
    """
    if _port_in_use(HTTP_HOST, HTTP_PORT):
        print(f"ℹ️  {HTTP_HOST}:{HTTP_PORT} 已有服务在运行，直接连它。")
        return None

    # mcp.http_app() 把 FastMCP 实例包成一个标准 ASGI 应用，再交给 uvicorn 跑。
    # 用 uvicorn.Server 而不是 uvicorn.run()，是为了拿到 started / should_exit 两个开关。
    app = mcp.http_app(transport="streamable-http", path=MCP_PATH)
    server = uvicorn.Server(
        uvicorn.Config(app, host=HTTP_HOST, port=HTTP_PORT, log_level="warning")
    )
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    for _ in range(100):     # 轮询等就绪，最多 10 秒；比写死 sleep 可靠
        if server.started:
            return server, thread
        time.sleep(0.1)
    print(f"❌ 服务端启动失败：{HTTP_HOST}:{HTTP_PORT} 无法监听。")
    return None


if __name__ == "__main__":
    # notebook 里用的是「独立进程 + 最后 taskkill」的姿势；这段入口是给
    # 「终端里直接 python server.py」准备的：单独跑时服务常驻，Ctrl+C 走下面的优雅退出。
    started = start_server_in_thread()
    try:
        if started:
            server, thread = started
            while thread.is_alive():
                thread.join(timeout=10)
    except KeyboardInterrupt:
        pass
    finally:
        if started:
            server, thread = started
            server.should_exit = True
            thread.join(timeout=10)
            print("\n✅ 本文件启动的服务端已关闭。")
'''

print(f"SERVER_MAIN 已就绪：{len(SERVER_MAIN.splitlines())} 行"
      "（端口常量、资源、提示词、脚手架四段齐了，下一格拼起来）")

### 预期输出

```text
SERVER_MAIN 已就绪：60 行（端口常量、资源、提示词、脚手架四段齐了，下一格拼起来）
```

三段文本（`SERVER_RESOURCES` + `SERVER_PROMPTS` + `SERVER_MAIN`）按顺序拼起来，就是一个完整的
**独立服务端程序**：它既能被 `python server.py` 单独跑，也能被本 notebook 用子进程拉起来。

In [ ]:
# ---------- 2.5 拼成 server.py（三段拼接 + 端口一致性断言 + 落盘）----------
SERVER_PY = MB_DIR / "server.py"
server_text = "\n".join([
    SERVER_RESOURCES.strip("\n"),
    SERVER_PROMPTS.strip("\n"),
    SERVER_MAIN.strip("\n"),
]) + "\n"

# 端口只允许一处不一致 → 宁可现在报错，也不要等客户端连到别的服务上去
assert f"HTTP_PORT = {HTTP_PORT}" in server_text, "脚本里的端口和 notebook 不一致，检查 2.2 那段常量"
assert "mcp.run()" not in server_text, "完整版脚本不能保留 mcp.run()：它会永久阻塞"

SERVER_PY.write_text(server_text, encoding="utf-8")
print(f"已写入：{SERVER_PY.relative_to(ROOT)}")
print(f"行数：{len(server_text.splitlines())}")
print(f"端口：{HTTP_HOST}:{HTTP_PORT}{MCP_PATH}")
print(f"装饰器数量：@mcp.resource "
      f"{sum(1 for ln in server_text.splitlines() if ln.startswith('@mcp.resource'))} 个 / "
      f"@mcp.prompt {sum(1 for ln in server_text.splitlines() if ln.startswith('@mcp.prompt'))} 个")

### 预期输出

```text
已写入：Agent\05_mcp\tmp_nb_work\mcp_resources\server.py
行数：355
端口：127.0.0.1:8120/mcp
装饰器数量：@mcp.resource 6 个 / @mcp.prompt 4 个
```

### 2.6 后台起服务：独立进程 + 端口轮询

这一格是本课的关键动作，两个细节：

1. **独立进程**（`subprocess.Popen`）而不是在本内核里起线程 —— 服务端崩了不会带崩 notebook，
   端口和进程都看得见（`pid`），最后也能干净地收掉整棵进程树；
2. **轮询端口就绪，而不是写死 `sleep`** —— 就绪了立刻往下走，最多等 10 秒；
   写死 `sleep(3)` 既可能不够、又白等。

`cwd=str(ROOT)` 是必须的：服务端脚本里要 `from config import settings`。

In [ ]:
# ---------- 2.6 后台起服务：独立进程 + 轮询等端口就绪 ----------
import subprocess

# 为什么不能直接在本格里 mcp.run()：那是常驻调用，会永久阻塞内核，后面每一格都不会执行。
server = subprocess.Popen(
    [sys.executable, str(SERVER_PY)],
    cwd=str(ROOT),                       # 服务端要 from config import settings，cwd 必须是仓库根
    env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8",
         "NO_PROXY": "127.0.0.1,localhost"},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
)

for _ in range(100):          # 轮询等待就绪，最多 10 秒（比写死 sleep 可靠）
    if _port_in_use(HTTP_HOST, HTTP_PORT):
        break
    time.sleep(0.1)

print("服务进程 pid：", server.pid)
print(f"端口 {HTTP_PORT} 就绪：", _port_in_use(HTTP_HOST, HTTP_PORT))
print("进程状态：", "运行中" if server.poll() is None else f"已退出（code={server.poll()}）")

# 进程要是已经退出了（= 起服务就失败），把它的输出读出来看真因 ——
# 进程已退出，read() 不会阻塞；这在排查「为什么端口一直不就绪」时是最有用的一条信息。
if server.poll() is not None:
    print("⚠️  服务端进程已退出，它自己的输出如下：")
    print(server.stdout.read() if server.stdout else "（没有输出）")

### 预期输出

```text
服务进程 pid： 50156
端口 8120 就绪： True
进程状态： 运行中
```

> `pid` 每次运行都不一样，这里贴的是本机实跑的那一次；
> 若服务端起不来（`进程状态：已退出`），下面会紧接着打印**服务端自己的输出**——
> 那才是真因，别去猜「端口是不是被占了」。

### 2.7 客户端①：列出并读取资源

客户端这边要注意四件事：

1. **固定资源和动态资源分两次列**：`list_resources()` 与 `list_resource_templates()`，
   后者的字段名是 `uriTemplate`；
2. **`read_resource()` 返回的是列表** —— 同一个 URI 可以对应多份内容，取正文要 `[0].text`；
3. **`db://tables` / `db://table/...` 是真查库**：本机 `langgraph` 库有 6 张表，
   `checkpoints` 表一行就有几千字符（存的是整个图状态），所以只取 1 行并截断打印；
4. **注入尝试**：`db://table/users;DROP/rows/1` 会被服务端的正则白名单直接挡掉，`DROP` 永远到不了数据库。

> 还有一处**本 notebook 的改写**：源文件末尾写的是 `asyncio.run(main(url))`，
> 但 notebook 内核里**已经有一个正在跑的事件循环**，直接调会报
> `RuntimeError: asyncio.run() cannot be called from a running event loop`，
> 所以这里改成**顶层 `await main(url)`**（IPython 的 autoawait，本机已实测）；
> 原写法留在一个 `if "__file__" in globals():` 分支里，保证这个 notebook 导出成 `.py`
> 单独跑时仍然可用。

In [ ]:
# ---------- 2.7 客户端①：列出资源 → 读固定 / 动态 / 多级动态 / 数据库资源 ----------
url = f"http://{HTTP_HOST}:{HTTP_PORT}{MCP_PATH}"


async def main(url: str) -> None:
    from fastmcp import Client

    async with Client(url) as client:
        # --- FastMCP 3.x 把资源分两类 ---
        # list_resources()           ：固定资源（URI 里没有 {参数}）
        # list_resource_templates()  ：动态资源（URI 里含 {参数}），字段名是 uriTemplate
        fixed = await client.list_resources()
        templates = await client.list_resource_templates()
        print(f"固定资源: {len(fixed)} 个")
        for r in fixed:
            print(f"  {r.uri}: {r.name} - {r.description}")
        print(f"动态资源: {len(templates)} 个")
        for t in templates:
            print(f"  {t.uriTemplate}: {t.name} - {t.description}")

        # --- 读取固定资源 ---
        # 注意 read_resource 返回的是**列表**（一段 URI 可以对应多份内容），所以下面要取 [0]
        config = await client.read_resource("config://app")
        print(f"\n配置信息：{config}")

        # --- 读取带参数的动态资源（把 {参数} 换成具体值） ---
        # 走的是真数据库查询；连不上库时服务端会降级成内存数据并打印中文提示
        top3 = await client.read_resource("users://top/3")
        print(f"\nTOP3 用户：{top3}")

        # --- 多级动态资源：api://users/{user_id}/posts/{post_id} ---
        # 这一段是为了演示课案 URI 设计规范里的第三种写法（scheme://{a}/{b}）
        post = await client.read_resource("api://users/u-1001/posts/p-07")
        print(f"\n多级动态资源：{post}")

        # --- 动态资源的另一种：命中 / 未命中各来一次 ---
        doc = await client.read_resource("file://docs/readme")
        print(f"\n文档 readme：{doc[0].text}")
        miss = await client.read_resource("file://docs/不存在")
        print(f"文档 不存在：{miss[0].text}")

        # --- 数据库元数据资源 ---
        # 真查 information_schema；.text 才是内容正文，直接 print 对象会看到一堆类型信息
        tables = await client.read_resource("db://tables")
        print(f"\n数据库表清单：\n{tables[0].text}")

        # --- 读具体表的前几行（真表；先看 db://tables 里有什么） ---
        # langgraph 的 checkpoints 表一行就有几千字符（存的是整个图状态），
        # 所以这里只取 1 行、并且打印时截断，避免刷屏。
        rows = await client.read_resource("db://table/checkpoints/rows/1")
        preview = rows[0].text
        print(f"\ncheckpoints 前 1 行（截断展示）：\n{preview[:280]}……（共 {len(preview)} 字符）")

        # --- 注入尝试会被挡下来（教学演示） ---
        # "users;DROP" 这种表名会被 _IDENT_RE 白名单直接拒绝，连数据库都不会碰
        evil = await client.read_resource("db://table/users;DROP/rows/1")
        print(f"\n注入尝试的返回：{evil[0].text}")


# 【notebook 改写】源文件末尾是 asyncio.run(main(url))：notebook 内核里已经有事件循环，
# 直接调会 RuntimeError（原因见 1.2），所以这里改成**顶层 await**。
await main(url)

# 下面这行只为「把本 notebook 导出成 .py 再单独跑」保留 —— 那个场景下 __file__ 才存在。
if "__file__" in globals():
    asyncio.run(main(url))

### 预期输出

```text
固定资源: 2 个
  config://app: get_app_config - 返回应用配置信息
  db://tables: list_db_tables - 列出当前数据库里所有业务表（真查 information_schema）
动态资源: 4 个
  file://docs/{filename}: get_file_content - 根据文件名返回文档内容
  api://users/{user_id}/posts/{post_id}: get_user_post - 多级动态资源示例：读取某个用户的某篇帖子
  users://top/{limit}: get_top_users - 查询排名前 N 的用户数据
  db://table/{table}/rows/{limit}: get_table_rows - 查看某张表的前 N 行（表名走严格白名单校验，防 SQL 注入）

配置信息：[TextResourceContents(uri=AnyUrl('config://app'), mimeType='text/plain', meta=None, text='\n    {\n      "app_name": "计算器服务",\n      "version": "1.0.0",\n      "max_precision": 10\n    }\n    ')]

TOP3 用户：[TextResourceContents(uri=AnyUrl('users://top/3'), mimeType='text/plain', meta=None, text='[{"name": "张三", "level": 99}, {"name": "李四", "level": 85}, {"name": "王五", "level": 72}]')]

多级动态资源：[TextResourceContents(uri=AnyUrl('api://users/u-1001/posts/p-07'), mimeType='text/plain', meta=None, text='{"user_id": "u-1001", "post_id": "p-07", "title": "u-1001 的第 p-07 篇帖子"}')]

文档 readme：# 计算器服务

提供基础数学运算的 MCP 服务。
文档 不存在：文件 '不存在' 不存在

数据库表清单：
{
  "count": 6,
  "tables": [
    {
      "schema": "public",
      "table": "checkpoint_blobs"
    },
    {
      "schema": "public",
      "table": "checkpoint_migrations"
    },
    {
      "schema": "public",
      "table": "checkpoint_writes"
    },
    {
      "schema": "public",
      "table": "checkpoints"
    },
    {
      "schema": "public",
      "table": "store"
    },
    {
      "schema": "public",
      "table": "store_migrations"
    }
  ]
}

checkpoints 前 1 行（截断展示）：
{
  "table": "checkpoints",
  "count": 1,
  "rows": [
    [
      "1",
      "",
      "1f1af7a5-ba6f-69af-bfff-a5f5515ac68b",
      null,
      null,
      {
        "v": 4,
        "id": "1f1af7a5-ba6f-69af-bfff-a5f5515ac68b",
        "ts": "2026-09-13T13:52:28.373649+00:00",
 ……（共 880 字符）

注入尝试的返回：表名非法：'users;DROP'。只允许字母/数字/下划线，且以字母或下划线开头。
```

> 「数据库表清单」和「checkpoints 前 1 行」的内容**取决于你库里真实的数据**
> （表名、行数、时间戳都会变），这里贴的是本机实跑的那一次；只要结构一样就说明跑对了。
> 最后那条注入尝试是重点：表名连数据库都没碰到就被白名单拒了。

### 2.8 客户端②：列出并获取提示词

三个要点：

1. **`list_prompts()` 只给「模板定义」**（名字 + 描述 + 参数表），**不含内容**，也不花任何模型调用；
   描述就是服务端函数的 `docstring`；
2. **`get_prompt(name, args)` 才是填参数、拼内容** —— 返回的是 `PromptMessage` 列表；
   FastMCP 3.x 取正文要多走一层：`prompt.messages[0].content.text`（老版本直接是 `str`）；
3. **带默认值的参数不用传**：`math_tutor` 只传 `question`，`difficulty` 由服务端填「中等」。

In [ ]:
# ---------- 2.8 客户端②：列出提示词 → 逐个填充参数拉下来看 ----------
async def main(url: str) -> None:
    from fastmcp import Client

    async with Client(url) as client:
        # --- 列出所有提示词模板 ---
        # list_prompts() 给的是「模板定义」：名字 + 描述 + 参数表，不含内容，也不花任何模型调用
        prompts = await client.list_prompts()
        print("可用提示词：")
        for p in prompts:
            print(f"  {p.name}: {p.description}")

        # --- 获取提示词（填充参数） ---
        # get_prompt 才是「填参数、拼内容」；返回的是 PromptMessage 列表
        prompt = await client.get_prompt(
            "code_review",
            {
                "language": "Python",
                "code": "def add(a,b):\n    return a+b",
            },
        )
        # FastMCP 3.x：.content 是 TextContent 对象，用 .text 取字符串
        # （课案在这里特意标注过：老版本直接是 str，新版本要多一层 .text）
        print(f"\n生成的提示词：\n{prompt.messages[0].content.text}")

        # --- 获取带角色设定的提示词 ---
        # style="幽默" 覆盖了函数签名里的默认值 "正式"；不传就用默认值
        prompt2 = await client.get_prompt(
            "writing_assistant",
            {"topic": "人工智能的未来", "style": "幽默"},
        )
        print(f"\n写作助手提示词：\n{prompt2.messages[0].content.text}")

        # --- 复杂模板：注意 difficulty 用默认值「中等」 ---
        # 只传必填参数也能成功 —— 这就是自定义提示词模板参数默认值的用途
        prompt3 = await client.get_prompt(
            "math_tutor", {"question": "为什么 0.999… = 1？"}
        )
        print(f"\n数学辅导提示词（difficulty 取默认值）：\n{prompt3.messages[0].content.text}")

        # --- 多角色模板：messages 里有两条，各带自己的 role ---
        # 这是课案没写、但工程里常用的写法：返回**消息列表**而不是一大段字符串。
        # 原版这里是 list[dict]（1.3 已经翻过车），完整版改成了 Message 对象。
        prompt4 = await client.get_prompt(
            "translate", {"text": "工具调用是 Agent 的基础能力。", "target_lang": "英文"}
        )
        print(f"\n翻译模板共 {len(prompt4.messages)} 条消息：")
        for m in prompt4.messages:
            print(f"  [{m.role}] {m.content.text}")


# 同样是顶层 await（原因见 2.7 的改写说明）
await main(url)

# 只为「导出成 .py 单独跑」保留的那一行
if "__file__" in globals():
    asyncio.run(main(url))

### 预期输出

```text
可用提示词：
  code_review: 代码审查提示词
  writing_assistant: 写作助手提示词，内含角色和写作要求
  math_tutor: 数学辅导提示词
  translate: 翻译提示词：用 assistant 消息做角色铺垫，再用 user 消息给任务

生成的提示词：
请审查以下 Python 代码，从以下几个方面给出建议：
1. 代码规范性
2. 性能优化
3. 安全漏洞
4. 可读性

代码：
def add(a,b):
    return a+b

写作助手提示词：
你是一位幽默风格的专业写手。
请写一篇关于「人工智能的未来」的文章，要求不少于500字。

数学辅导提示词（difficulty 取默认值）：
你是一位耐心的数学老师，学生提出了以下问题：

为什么 0.999… = 1？

难度等级：中等

要求：
- 先给出解题思路，再给出详细步骤
- 用通俗易懂的语言讲解
- 如果难度为"困难"，需要补充相关的知识点背景

翻译模板共 2 条消息：
  [assistant] 我是专业翻译，译文自然流畅，只输出译文本身。
  [user] 把下面的内容翻译成英文：
工具调用是 Agent 的基础能力。
```

> 三个对照点：① `code_review` 是**必填两个参数**的模板；
> ② `writing_assistant` 传了 `style="幽默"`，覆盖服务端默认值「正式」；
> ③ `math_tutor` **只传了 `question`**，`difficulty` 由服务端填「中等」——
> 参数默认值就是提示词模板「统一规范、客户端少写代码」的地方。

### 2.9 收尾：关掉后台服务

这是**最后一个格子**，它必须做完三件事：

1. `taskkill /F /T /PID <pid>` —— Windows 上要连**整棵子进程树**一起收（`/T`），
   只 kill 父进程可能留下 uvicorn 的兄弟进程占着端口，下一轮就会连到旧实例上；
2. **等它真的退出**再报状态 —— 刚 kill 完 `poll()` 还是 `None`，端口也不是立刻释放；
3. 打印进程状态与端口状态，作为「服务确实关了」的证据
   （`已结束（code=1）` 里的 `code=1` 是 taskkill 强杀的正常结果，不是报错）。

> **为什么不把服务端 stderr 全打印出来**：Windows 上 uvicorn 0.52 会硬选
> `ProactorEventLoop`（`uvicorn.loops.asyncio.asyncio_loop_factory` 里写死的，
> 连 `asyncio.set_event_loop_policy()` 都改不动），客户端连接被强制关闭时它会在 stderr
> 喷一串 `_ProactorBasePipeTransport._call_connection_lost` + `ConnectionResetError` 的回调噪音。
> 那串东西与业务无关、却会淹掉整格输出，所以这里**只在「服务端自己先退出了」**
> （即起服务真的失败）时才把管道读出来。

In [ ]:
# ---------- 2.9 收尾：关掉后台服务（连子进程树一起收）----------
# 服务端 stderr 都收在管道里。正常运行时它没有输出；万一它自己先退出了，这时才读它：
# 进程已退出，read() 不会阻塞。
if server.poll() is not None:
    print("⚠️  服务端在此之前已自行退出，它的输出如下：")
    print((server.stdout.read() if server.stdout else "").strip() or "（没有输出）")

if server.poll() is None:
    subprocess.run(["taskkill", "/F", "/T", "/PID", str(server.pid)],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)

# 等它真的退出：进程还活着时既不能读管道，端口也不会立刻释放
for _ in range(40):
    if server.poll() is not None:
        break
    time.sleep(0.25)

print("进程状态：", "仍在运行" if server.poll() is None else f"已结束（code={server.poll()}）")
print(f"端口 {HTTP_PORT} 仍在监听：", _port_in_use(HTTP_HOST, HTTP_PORT))
print("\n✅ 本文件启动的服务端已关闭。")

### 预期输出

```text
进程状态： 已结束（code=1）
端口 8120 仍在监听： False

✅ 本文件启动的服务端已关闭。
```

> `code=1` 是 `taskkill /F` **强杀**的结果，不是报错；关键是最后一行「端口仍在监听： False」——
> 端口真的释放了，下一轮（或同一个 notebook 再跑一遍）才不会连到旧实例上。

## 小结

- **工具 / 资源 / 提示词**是三件正交的事：做事（有副作用）、读数据（只读）、发模板。
  资源之所以敢用 `config://app`、`file://docs/{name}` 这种「像文件路径」的寻址，正因为它只读。
- **资源的核心是 URI 而不是函数名**：固定资源进 `list_resources()`，
  带 `{}` 的动态资源进 `list_resource_templates()`，模板里参数名必须和函数参数名一致。
- **`read_resource()` 返回列表**，正文在 `[0].text`；返回 `str` 是 `text/plain`，
  返回 `dict` 才自动标 `application/json`。
- **数据库类资源要三件事一起做**：连接串从 `settings` 拼（不硬编码）、
  失败降级成中文提示 + 演示数据（不让客户端看到一个连接异常）、
  表名走白名单 + 存在性双重校验（标识符拼串就是注入点）。
- **提示词服务化的价值在「统一分发」**：`list_prompts()` 看模板，`get_prompt()` 填参数，
  参数默认值由服务端兜底，Prompt 本身不调用模型。
- **FastMCP 3.x 的两处形状变化**：多角色提示词必须用 `Message` 包装（`list[dict]` 会报错）；
  `Message.role` 只有 `user` / `assistant`（没有 `system`）。
- **常驻服务的正确姿势**：原版 stdio（客户端拉子进程）随用随停；
  完整版 HTTP 自己起服务 —— 起的时候后台轮询端口、关的时候 `taskkill /F /T` 收进程树。
- **本 notebook 的两处必要改写**（都是「把脚本搬进 notebook」逼出来的，原因写在对应格子的注释里）：
  ① 常驻服务从「进程内 `mcp.run()`」改成「**独立进程** + 末尾 `taskkill`」；
  ② `asyncio.run(main(url))` 改成**顶层 `await main(url)`**（内核里已有事件循环），
  原写法保留在 `if "__file__" in globals():` 分支里，保证导出成 `.py` 还能单独跑。

## 常见坑

| 症状 | 原因 | 解法 |
|---|---|---|
| `messages[0] must be Message or str, got dict` | 提示词返回了 `list[dict]`（课案原版写法） | 用 `fastmcp.prompts.Message(role=..., content=...)` 包装每一条 |
| 提示词里的 `system` 角色被拒 | MCP 的 `PromptMessage.role` 只有 `user` / `assistant` | 角色折进正文，或用 `assistant` 先说一句铺垫 |
| 内核卡住不再往下跑 | 直接把 `mcp.run()` / `uvicorn.run()` 放进格子 | 改成 `subprocess.Popen` 起独立进程，最后一个格子 `taskkill /F /T` |
| `ModuleNotFoundError: config` | 服务端子进程没站在仓库根 | `subprocess.Popen(..., cwd=str(ROOT))`，或 notebook 首格 bootstrap |
| 服务端日志把 notebook 刷屏 / 协议乱掉 | stdio 的 stdout 是协议通道 | `PythonStdioTransport(..., log_file=...)` 把 stderr 写进文件 |
| 客户端 `Connection closed` | 服务端脚本一启动就崩了（如少 `import sys`） | 去看 `log_file` 里的完整 traceback，别猜 |
| 端口被占 / 连到旧实例 | 上一轮的服务没关干净 | `taskkill /F /T /PID`；起服务前用 `_port_in_use()` 先探一下 |
| `AttributeError: 'ResourceTemplate' object has no attribute 'uri'` | 动态资源的字段名是 `uriTemplate` | 固定资源用 `.uri`，模板资源用 `.uriTemplate` |
| 表名带 `;` 之类的字符 | 表名是标识符、不能走 `%s` 占位，只能拼串 | 先正则白名单 + `information_schema` 存在性校验，再拼串 |
| 库没起时客户端一直干等 | 连接没设超时 | `psycopg.connect(..., connect_timeout=3)` |
| `RuntimeError: asyncio.run() cannot be called from a running event loop` | notebook 内核里已经有事件循环 | 改成**顶层 `await xxx()`**；`asyncio.run(...)` 只留给「导出成 .py 单独跑」 |
| Windows 上服务端 stderr 一串 `_ProactorBasePipeTransport` + `ConnectionResetError` | uvicorn 0.52 在 Windows 硬选 `ProactorEventLoop`，客户端被强杀时回调报错 | 与业务无关的噪音：别把它当故障；真要干净就换 `SelectorEventLoop` 或别回显这段 stderr |

## 官方链接

- MCP 规范 · Resources（URI、资源模板、`read_resource` 的返回结构）：
  <https://modelcontextprotocol.io/specification/2026-07-28/server/resources>
- MCP 规范 · Prompts（`PromptMessage` 的 `role` 只允许 user / assistant 就在这一页）：
  <https://modelcontextprotocol.io/specification/2026-07-28/server/prompts>
- FastMCP 文档 · Resources & Templates（`@mcp.resource` / URI 模板）：
  <https://gofastmcp.com/servers/resources>
- FastMCP 文档 · Prompts（`Message`、参数默认值）：
  <https://gofastmcp.com/servers/prompts>
- FastMCP 文档 · Client Transports（stdio / http 两种传输的取舍）：
  <https://gofastmcp.com/clients/transports>